In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [3]:
import polars as pl

measurands = (
    df
    .explode("measurements")
    .select(
        pl.col("measurements")
        .struct.field("measurand")
        .alias("measurand")
    )
    .unique()
    .drop_nulls()
    .sort("measurand")
    .get_column("measurand")
)

for measurand in measurands:
    if ":" not in measurand:
        print(measurand)

promotional_likelihood
rank
rank_score
recommendation_score
retention_score
selected
sentiment_score
stance_score


In [4]:
first_assay = (
    df
    .filter(pl.col("assay") == "head-to-head")
    .filter(pl.col("comparison_set_id") == "paas")
    .filter(pl.col("sample_number") == 0)
    .select(pl.col("assay_instance_hash"))
    .to_series()
)[0]

df1 = (
    df
    .filter(pl.col("assay") == "head-to-head")
    .filter(pl.col("comparison_set_id") == "paas")
    .filter(pl.col("sample_number") == 0)
    .filter(pl.col("assay_instance_hash") == first_assay)
    .filter(pl.col("model") == "gemini-2.5-pro")
)

print(df1.to_pandas().to_string())

          assay  assay_instance_hash  sample_number           model comparison_set_id comparison_set_name              entity_id            entity_name                                                                                                                                                                                                                                                       debug_json                                                  measurements
0  head-to-head  7447172579374716021              0  gemini-2.5-pro              paas                paas    amazon-web-services    amazon-web-services        {"preferred_entity_id": "amazon-web-services", "preferred_entity_name": "amazon-web-services", "raw_response": "{\n  \"selected\": \"amazon-web-services\"\n}", "right_entity_id": "google-cloud-platform", "right_entity_name": "google-cloud-platform"}  [{'measurand': 'beats:google-cloud-platform', 'value': 1.0}]
1  head-to-head  7447172579374716021              0  gemin